# CSoT'26 - ML in Astronomy - Week 2 . Part 1: Baseline with Scikit-Learn (Starter)

**Goal:** Flatten the galaxy images into NumPy feature rows, train simple scikit-learn classifiers (KNN, Logistic Regression), and record the **baseline accuracy** that every later model must beat.

**Before you begin:**
1. Switch this notebook to a **GPU runtime** (`Runtime -> Change runtime type -> GPU`). The GPU isn't strictly needed for sklearn, but we reuse the Week 1 GPU pipeline.
2. Read [`01-tensors-to-numpy-and-flattening.md`](../01-tensors-to-numpy-and-flattening.md) and [`02-baseline-with-scikit-learn.md`](../02-baseline-with-scikit-learn.md).

Each `TODO` cell has a short instruction. Replace the placeholder with working code, then run the cell. **Do not** open the solution until you've genuinely attempted every TODO.

## Step 0 — Re-create the Week 1 data pipeline

Week 2 builds directly on the `DataLoader`s from Week 1. The cells below reproduce that pipeline (download is commented out — uncomment it the first time, exactly as in [`week1_data_solution.ipynb`](../../Week-1/notebooks/week1_data_solution.ipynb)). If you saved `galaxy_data/` to Google Drive in Week 1, just re-mount Drive and point `DATA_ROOT` at it instead of re-downloading.

After this section you should have `train_loader`, `val_loader`, `test_loader`, `train_ds`, and `num_classes`.

In [2]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", device)

Using: cuda


## Step 1 - From DataLoader to NumPy feature matrices

scikit-learn wants a 2D array `X` of shape `(n_samples, n_features)` and a 1D array `y` of labels. We get there by iterating the loader, flattening each batch with `flatten(start_dim=1)` (keeping the batch dim), and concatenating.

In [1]:
# TODO: paste your Week 1 data pipeline here so that the following names are defined:
#   train_ds, val_ds, test_ds, train_loader, val_loader, test_loader, num_classes
#
# The quickest path is to copy the data-prep cells from
# ../../Week-1/notebooks/week1_data_solution.ipynb (Steps 1-8), then add:
#   num_classes = len(train_ds.classes)

import os
import json
import shutil
from pathlib import Path
from torchvision import transforms # Added this line

# --- 1. KAGGLE AUTHENTICATION ---
# Replace with your EXACT username and key from your Kaggle account
kaggle_creds = {
  "username": "your_username_here",
  "key": "your_long_alphanumeric_key_here"
}

# Create the hidden Kaggle directory and write the credentials file
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(kaggle_creds, f)

# Lock down the file permissions for security
!chmod 600 /root/.kaggle/kaggle.json

# --- 2. DIRECTORY SETUP ---
RAW_ROOT = Path("galaxy_raw")
IMAGES_DIR = RAW_ROOT / "images_gz2"
DATA_ROOT = Path("galaxy_data")
RAW_ROOT.mkdir(parents=True, exist_ok=True)

# --- 3. DOWNLOAD KAGGLE IMAGES ---
print("Authenticating and downloading images from Kaggle. This might take a minute...")
dataset_name = "jaimetrickz/galaxy-zoo-2-images"
!kaggle datasets download -d {dataset_name} -p {RAW_ROOT} --unzip

# --- 4. DOWNLOAD HART LABELS ---
print("Downloading Hart et al. morphology labels...")
!wget -q -O {RAW_ROOT}/gz2_hart16.csv.gz https://gz2hart.s3.amazonaws.com/gz2_hart16.csv.gz
!gunzip -f {RAW_ROOT}/gz2_hart16.csv.gz

# --- 5. FIX NESTED DIRECTORY ---
# Kaggle sometimes double-wraps the extracted folder. This flattens it.
nested_dir = IMAGES_DIR / "images_gz2"

if nested_dir.exists() and nested_dir.is_dir():
    print("Flattening the nested image directory...")
    for file in nested_dir.glob("*.jpg"):
        shutil.move(str(file), str(IMAGES_DIR))
    nested_dir.rmdir() # Clean up the empty duplicate folder

import shutil
from pathlib import Path

RAW_ROOT = Path("galaxy_raw")
IMAGES_DIR = RAW_ROOT / "images_gz2"

print("Hunting for the hidden images...")

# Recursively find ALL jpg files anywhere inside RAW_ROOT
all_jpgs = list(RAW_ROOT.rglob("*.jpg"))
print(f"Found {len(all_jpgs)} total images hiding in the raw folder!")

if len(all_jpgs) > 0:
    print("Moving them to the correct IMAGES_DIR... (This might take a minute)")

    # Ensure IMAGES_DIR exists
    IMAGES_DIR.mkdir(parents=True, exist_ok=True)

    # Move them all to the flat directory
    moved_count = 0
    for img_path in all_jpgs:
        # Only move if it's not already in the right spot
        if img_path.parent != IMAGES_DIR:
            shutil.move(str(img_path), str(IMAGES_DIR / img_path.name))
            moved_count += 1

    print(f"Successfully moved {moved_count} images!")

# Final Recount
print("\n--- Final Image Recount ---")
image_count = len(list(IMAGES_DIR.glob('*.jpg')))
print(f"Total images exactly in target folder: {image_count}")

if image_count > 240000:
    print("✅ Found them! Data pipeline setup successful! Ready for Step 2.")
else:
    print("⚠️ Warning: Something is still odd. Let me know what it prints!")



# --- 6. FINAL SANITY CHECK ---
print("\n--- Final Download Check ---")
print(f"Mapping CSV exists: {(RAW_ROOT / 'gz2_filename_mapping.csv').exists()}")
print(f"Hart labels CSV exists: {(RAW_ROOT / 'gz2_hart16.csv').exists()}")
print(f"Images folder exists: {IMAGES_DIR.exists()}")

if IMAGES_DIR.exists():
    image_count = len(list(IMAGES_DIR.glob('*.jpg')))
    print(f"Total images found: {image_count}")
    if image_count > 240000:
        print("✅ Data pipeline setup successful! Ready for Step 2.")
    else:
        print("⚠️ Warning: Image count seems low.")



Authenticating and downloading images from Kaggle. This might take a minute...
Dataset URL: https://www.kaggle.com/datasets/jaimetrickz/galaxy-zoo-2-images
License(s): Attribution 4.0 International (CC BY 4.0)
100% 3.06G/3.06G [00:34<00:00, 96.2MB/s]

Hunting for the hidden images...
Found 243434 total images hiding in the raw folder!
Moving them to the correct IMAGES_DIR... (This might take a minute)
Successfully moved 243434 images!

--- Final Image Recount ---
Total images exactly in target folder: 243434
✅ Found them! Data pipeline setup successful! Ready for Step 2.

--- Final Download Check ---
Mapping CSV exists: True
Hart labels CSV exists: True
Images folder exists: True
Total images found: 243434
✅ Data pipeline setup successful! Ready for Step 2.
--- 1. Contents of RAW_ROOT ---
 - images_gz2
 - gz2_hart16.csv
 - gz2_filename_mapping.csv

--- 2. Peek at IMAGES_DIR ---
 - 24167.jpg
 - 4578.jpg
 - 241975.jpg
 - 213062.jpg
 - 41099.jpg

--- 3. Checking gz2_filename_mapping.csv -

,objid,sample,asset_id
0,587722981736120347,original,1
1,587722981736579107,original,2
2,587722981741363294,original,3



✅ Success: Expected columns (objid, sample, asset_id) are all present!
RAW_ROOT contents: ['gz2_filename_mapping.csv', 'gz2_hart16.csv', 'images_gz2']
Flat JPG count in galaxy_raw/images_gz2: 243,434

Mapping CSV preview:
                objid    sample  asset_id
0  587722981736120347  original         1
1  587722981736579107  original         2
2  587722981741363294  original         3

Labels CSV preview (note dr7objid — we rename to objid before merging):
             dr7objid gz2_class
0  587732591714893851      Sc+t
1  588009368545984617      Sb+t
2  587732484359913515        Ei
Joined rows: 239100

Label counts:
label
elliptical       97670
spiral           95849
spiral_barred    45581
Name: count, dtype: int64

Example rows:
   asset_id               objid gz2_class       label
0         3  587722981741363294        Sb      spiral
1         4  587722981741363323      Sc?l      spiral
2         5  587722981741559888        Er  elliptical
3         6  587722981741625481      Sc1t

NameError: name 'transforms' is not defined

In [3]:
# TODO: print os.listdir(RAW_ROOT) and count a few files in IMAGES_DIR.
#       Hint: Path(IMAGES_DIR).glob("*.jpg")
import pandas as pd
from pathlib import Path

# Assuming these are still set from the previous cell
RAW_ROOT = Path("galaxy_raw")
IMAGES_DIR = RAW_ROOT / "images_gz2"

print("--- 1. Contents of RAW_ROOT ---")
# List everything sitting in the raw folder
for item in RAW_ROOT.iterdir():
    print(f" - {item.name}")

print("\n--- 2. Peek at IMAGES_DIR ---")
# Grab just the first 5 jpg files to verify the naming convention
sample_images = list(IMAGES_DIR.glob('*.jpg'))[:5]
for img in sample_images:
    print(f" - {img.name}")

print("\n--- 3. Checking gz2_filename_mapping.csv ---")
mapping_csv_path = RAW_ROOT / "gz2_filename_mapping.csv"

if mapping_csv_path.exists():
    # Load the CSV into a pandas DataFrame
    df_mapping = pd.read_csv(mapping_csv_path)

    print("Columns found:")
    print(df_mapping.columns.tolist())

    print("\nFirst 3 rows:")
    # Display the first few rows to see the actual data
    display(df_mapping.head(3))

    # Programmatic check for the specific columns
    expected_cols = {'objid', 'sample', 'asset_id'}
    actual_cols = set(df_mapping.columns)

    if expected_cols.issubset(actual_cols):
        print("\n✅ Success: Expected columns (objid, sample, asset_id) are all present!")
    else:
        print("\n⚠️ Warning: Missing expected columns.")
else:
    print(f"⚠️ Error: Could not find {mapping_csv_path}")


print("RAW_ROOT contents:", sorted(p.name for p in RAW_ROOT.iterdir()))
jpg_count = sum(1 for _ in IMAGES_DIR.glob("*.jpg"))
print(f"Flat JPG count in {IMAGES_DIR}: {jpg_count:,}")

print("\nMapping CSV preview:")
print(pd.read_csv(RAW_ROOT / "gz2_filename_mapping.csv", nrows=3))

print("\nLabels CSV preview (note dr7objid — we rename to objid before merging):")
print(pd.read_csv(RAW_ROOT / "gz2_hart16.csv", nrows=3)[["dr7objid", "gz2_class"]])

def high_level_label(gz2_class: str):
    """Collapse detailed GZ2 codes (Sc2t, Ei, SBb2m, …) to a few training buckets."""
    if not isinstance(gz2_class, str) or gz2_class == "A":
        return None  # artifact / ambiguous
    if gz2_class.startswith("E"):
        return "elliptical"
    if gz2_class.startswith("SB"):
        return "spiral_barred"
    if gz2_class.startswith("S"):
        return "spiral"
    return None


def load_labeled_table(mapping_csv, labels_csv):
    """Join Kaggle mapping (objid ↔ asset_id) with Hart et al. morphology labels."""
    mapping = pd.read_csv(mapping_csv)
    labels = pd.read_csv(labels_csv)
    if "dr7objid" in labels.columns:
        labels = labels.rename(columns={"dr7objid": "objid"})
    df = mapping.merge(labels[["objid", "gz2_class"]], on="objid", how="inner")
    df["label"] = df["gz2_class"].map(high_level_label)
    df = df.dropna(subset=["label"]).reset_index(drop=True)
    return df


def _link_image(src: Path, dst: Path) -> bool:
    """Symlink if possible; otherwise copy (some Drive setups block symlinks)."""
    if dst.exists():
        return False
    dst.parent.mkdir(parents=True, exist_ok=True)
    try:
        os.symlink(src.resolve(), dst)
    except OSError:
        import shutil
        shutil.copy2(src, dst)
    return True


def build_split_imagefolder_layout(
    images_dir,
    df,
    out_root,
    per_class=200,
    train_frac=0.70,
    val_frac=0.15,
    test_frac=0.15,
    seed=42,
):
    """Create out_root/{train,val,test}/<class>/*.jpg for ImageFolder."""
    assert abs(train_frac + val_frac + test_frac - 1.0) < 1e-6
    images_dir = Path(images_dir)
    out_root = Path(out_root)
    summary = {}

    for label in sorted(df["label"].unique()):
        rows = df[df["label"] == label].sample(frac=1, random_state=seed)
        if len(rows) > per_class:
            rows = rows.head(per_class)

        n = len(rows)
        n_train = int(train_frac * n)
        n_val = int(val_frac * n)
        n_test = n - n_train - n_val
        splits = {
            "train": rows.iloc[:n_train],
            "val": rows.iloc[n_train : n_train + n_val],
            "test": rows.iloc[n_train + n_val :],
        }

        summary[label] = {}
        for split_name, split_rows in splits.items():
            linked = 0
            for _, row in split_rows.iterrows():
                src = images_dir / f"{int(row.asset_id)}.jpg"
                dst = out_root / split_name / label / f"{int(row.asset_id)}.jpg"
                if src.exists() and _link_image(src, dst):
                    linked += 1
            summary[label][split_name] = linked
    return summary

df = load_labeled_table(
    RAW_ROOT / "gz2_filename_mapping.csv",
    RAW_ROOT / "gz2_hart16.csv",
)
print("Joined rows:", len(df))
print("\nLabel counts:")
print(df["label"].value_counts())
print("\nExample rows:")
print(df[["asset_id", "objid", "gz2_class", "label"]].head())


PER_CLASS = 200  # increase once the pipeline works (e.g. 2000)

summary = build_split_imagefolder_layout(
    IMAGES_DIR,
    df,
    DATA_ROOT,
    per_class=PER_CLASS,
    train_frac=0.70,
    val_frac=0.15,
    test_frac=0.15,
    seed=42,
)
print("Linked images per class and split:")
print(pd.DataFrame(summary).fillna(0).astype(int))

for split in ("train", "val", "test"):
    split_dir = DATA_ROOT / split
    classes = sorted(p.name for p in split_dir.iterdir() if p.is_dir()) if split_dir.exists() else []
    n_imgs = sum(1 for _ in split_dir.rglob("*.jpg")) if split_dir.exists() else 0
    print(f"{split:5s}: {n_imgs:4d} images  classes={classes}")


transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

train_ds = ImageFolder(root=DATA_ROOT / "train", transform=transform)
val_ds   = ImageFolder(root=DATA_ROOT / "val",   transform=transform)
test_ds  = ImageFolder(root=DATA_ROOT / "test",  transform=transform)

for name, ds in [("train", train_ds), ("val", val_ds), ("test", test_ds)]:
    print(f"{name:5s}  n={len(ds):4d}  classes={ds.classes}")

print("class_to_idx:", train_ds.class_to_idx)


train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

images, labels = next(iter(train_loader))
print("train batch images:", images.shape)   # (32, 3, 64, 64)
print("train batch labels:", labels.shape)     # (32,)

--- 1. Contents of RAW_ROOT ---
 - images_gz2
 - gz2_hart16.csv
 - gz2_filename_mapping.csv

--- 2. Peek at IMAGES_DIR ---
 - 24167.jpg
 - 4578.jpg
 - 241975.jpg
 - 213062.jpg
 - 41099.jpg

--- 3. Checking gz2_filename_mapping.csv ---
Columns found:
['objid', 'sample', 'asset_id']

First 3 rows:


,objid,sample,asset_id
0,587722981736120347,original,1
1,587722981736579107,original,2
2,587722981741363294,original,3



✅ Success: Expected columns (objid, sample, asset_id) are all present!
RAW_ROOT contents: ['gz2_filename_mapping.csv', 'gz2_hart16.csv', 'images_gz2']
Flat JPG count in galaxy_raw/images_gz2: 243,434

Mapping CSV preview:
                objid    sample  asset_id
0  587722981736120347  original         1
1  587722981736579107  original         2
2  587722981741363294  original         3

Labels CSV preview (note dr7objid — we rename to objid before merging):
             dr7objid gz2_class
0  587732591714893851      Sc+t
1  588009368545984617      Sb+t
2  587732484359913515        Ei
Joined rows: 239100

Label counts:
label
elliptical       97670
spiral           95849
spiral_barred    45581
Name: count, dtype: int64

Example rows:
   asset_id               objid gz2_class       label
0         3  587722981741363294        Sb      spiral
1         4  587722981741363323      Sc?l      spiral
2         5  587722981741559888        Er  elliptical
3         6  587722981741625481      Sc1t

In [12]:
# TODO: write loader_to_numpy(loader) that returns (X, y) as NumPy arrays.
#   - for each (images, labels) batch: flat = images.flatten(start_dim=1)
#   - collect flat.numpy() and labels.numpy(), then np.concatenate along axis 0
# Then build X_train, y_train from train_loader and X_test, y_test from test_loader.
# Print X_train.shape and X_test.shape (expect (N, 12288)).

import numpy as np

def loader_to_numpy(loader):
    xs, ys = [], []
    for images, labels in loader:               # images: (B, 3, 64, 64)
        flat = images.flatten(start_dim=1)       # (B, 12288)
        xs.append(flat.numpy())
        ys.append(labels.numpy())
    X = np.concatenate(xs, axis=0)               # (N, 12288)
    y = np.concatenate(ys, axis=0)               # (N,)
    return X, y

X_train, y_train = loader_to_numpy(train_loader)
X_test,  y_test  = loader_to_numpy(test_loader)
print(X_train.shape, y_train.shape)
print(X_test.shape,y_test.shape)           # (N_train, 12288) (N_train,)
#print(X_train)

(420, 12288) (420,)
(90, 12288) (90,)


## Step 2 - The 'do-nothing' floors

Before any real model: the majority-class baseline. A model that always predicts the most common class already scores this much - so this, not random chance, is the number to beat.

In [14]:
# TODO: fit a DummyClassifier(strategy="most_frequent") on (X_train, y_train)
#       and print its accuracy on (X_test, y_test). Also print 1/num_classes (random).

from sklearn.dummy import DummyClassifier

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)
print("Majority-class accuracy:", dummy.score(X_test, y_test))

num_classes = len(set(y_train))

print("Random-guess accuracy:", 1 / num_classes)

Majority-class accuracy: 0.3333333333333333
Random-guess accuracy: 0.3333333333333333


## Step 3 - K-Nearest Neighbours

KNN classifies a galaxy by majority vote of its `k` closest training galaxies in 12 288-D pixel space. There is no real 'training' - it just memorises the data.

In [15]:
# TODO: fit KNeighborsClassifier(n_neighbors=5) on the train set and print its TEST accuracy.

from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5)   # k = 5
knn.fit(X_train, y_train)                    # just memorises the data
acc = knn.score(X_test, y_test)              # fraction correct on the test set
print(f"KNN accuracy: {acc:.3f}")

KNN accuracy: 0.411


## Step 4 - Logistic Regression

A linear classifier: a weighted sum of the 12 288 pixel features per class, squashed by a softmax. Effectively a single-layer neural network. `max_iter` is raised because high-dimensional data is slow to converge.

In [16]:
# TODO: fit LogisticRegression(max_iter=1000) on the train set and print its TEST accuracy.

from sklearn.linear_model import LogisticRegression

logreg = LogisticRegression(max_iter=1000)   # more iters: high-dim data is slow to converge
logreg.fit(X_train, y_train)
acc = logreg.score(X_test, y_test)
print(f"Logistic Regression accuracy: {acc:.3f}")

Logistic Regression accuracy: 0.467


## Step 5 - The comparison table (the bar to beat)

Put the numbers side by side. The best of these is the baseline your Week 3 CNN must clearly beat.

In [17]:
# TODO: print a small table comparing: random (1/num_classes), majority, KNN, LogReg.
#       Identify (in a print statement) the single number Week 3 must beat.
random_acc = 1 / num_classes
majority_acc = dummy.score(X_test, y_test)
knn_acc = knn.score(X_test, y_test)
logreg_acc = logreg.score(X_test, y_test)

print("\nModel Comparison")
print("-" * 35)
print(f"{'Random':<15} {random_acc:.4f}")
print(f"{'Majority':<15} {majority_acc:.4f}")
print(f"{'KNN':<15} {knn_acc:.4f}")
print(f"{'LogReg':<15} {logreg_acc:.4f}")

best_baseline = max(random_acc, majority_acc, knn_acc, logreg_acc)

print("\nWeek 3 must beat:", round(best_baseline, 4))



Model Comparison
-----------------------------------
Random          0.3333
Majority        0.3333
KNN             0.4111
LogReg          0.4667

Week 3 must beat: 0.4667


## Step 6 (stretch) - Confusion matrix

Accuracy hides *which* classes get confused. Plot a confusion matrix for the logistic-regression predictions and note the most-confused pair - we'll compare it to the CNN's matrix in Week 3.

In [ ]:
# TODO (optional): plot a confusion matrix for KNN or LogReg predictions.



## Reflection *(write 2-3 sentences each)*

1. What baseline accuracy did you reach, and how far above the majority-class floor is it?
2. Why does flattening hurt a *galaxy* classifier specifically? (Hint: isophote shape, arms - see [`03-surface-brightness-and-isophotes.md`](../03-surface-brightness-and-isophotes.md).)
3. Which two classes do you expect a CNN to still find hardest to separate, and why?

*(Replace this prompt with your answers.)*